In [0]:
customers='customers'
payments='payments'
vehicles='vehicles'
stations='stations'
sessions='sessions'
bronze='bronze'
silver='silver'
gold='gold'
catalog='voltgrid'


In [0]:
%sql
CREATE TABLE IF NOT EXISTS voltgrid.source.meta_trigger_data

In [0]:
%sh
pwd

In [0]:
%sql
DROP TABLE IF EXISTS voltgrid.default.bronze_customers

In [0]:
for s in spark.streams.active:
  s.stop()

In [0]:
%sql
DROP VOLUME IF EXISTS voltgrid.bronze.bronze_customer;

In [0]:
spark.sql("DROP TABLE IF EXISTS voltgrid.silver.customers")

In [0]:
%sql
SHOW VOLUMES IN voltgrid.bronze;

In [0]:
%sql
DROP TABLE IF EXISTS voltgrid.bronze.payments

In [0]:
%sql
SHOW TABLES IN voltgrid.silver;

In [0]:
spark.sql("DELETE FROM  voltgrid.bronze.customers")
spark.sql("VACUUM  voltgrid.bronze.customers RETAIN 1 HOURS")

In [0]:
%sql
select * from voltgrid.silver.payments where active_flg='Y';

In [0]:
%sql
SELECT session_id, customer_id, station_id, vehicle_id, payment_id
FROM voltgrid.silver.sessions
WHERE session_id IN ('CS-AU-00199694','CS-AU-00199695','CS-AU-00199696','CS-AU-00199697','CS-AU-00199698');

In [0]:
%sql
select *
from voltgrid.silver.vehicles;

In [0]:
%sql
select * from voltgrid.silver.stations

In [0]:
%sql
update voltgrid.silver.sessions
set customer_id ='CUST-AU-0199991'
where customer_id='CUST-AU-0073474'

In [0]:
%sql
select * 
from voltgrid.silver.sessions as a join voltgrid.silver.customers as b on a.customer_id=b.customer_id
where a.active_flg='Y';

In [0]:
%sql
select * from voltgrid.bronze.customers;

In [0]:
%sql
select * from voltgrid.silver.stations where active_flg='Y' limit 5;

In [0]:
%sql
select p.session_id
from voltgrid.silver.payments as p inner join voltgrid.silver.sessions  as s on p.session_id = s.session_id

In [0]:
%sql
update voltgrid.silver.sessions
set active_flg='Y'
where active_flg is null

In [0]:
%sql
DESCRIBE EXTENDED voltgrid.silver.customers;

In [0]:
%sql
ALTER TABLE voltgrid.silver.sessions
ADD COLUMNS(end_date date,start_date date);
---CLUSTER BY (station_id,state_code,charger_type,city)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS voltgrid.silver.payments (
    id STRING,
    payment_id STRING,
    session_id STRING,
    customer_id STRING,
    gateway STRING,
    amount_aud DECIMAL(18,2),
    gst DECIMAL(18,2),
    payment_mode STRING,
    processed_at TIMESTAMP,
    status STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    ingestion_time TIMESTAMP,
    batch_no INT,
    source_path STRING
)
USING DELTA
LOCATION 'abfss://silver@voltgridstorageaccount.dfs.core.windows.net/payments/'
CLUSTER BY (customer_id, status, processed_at);

In [0]:
%sql
DESCRIBE EXTENDED voltgrid.silver.stations

In [0]:
%sql
select * from voltgrid.silver.stations where active_flg='Y'

In [0]:
%sql
select * from voltgrid.bronze.sessions;

In [0]:
spark.sql("""
          CREATE TABLE IF NOT EXISTS voltgrid.bronze.stations (
              id INT,
              station_id STRING,
              name STRING,
              state_code STRING,
              city STRING,
              latitude STRING,
              longitude STRING,
              charger_type STRING,
              max_power_kw STRING,
              num_connectors INT,
              operator STRING,
              is_active BOOLEAN,
              commissioned_date STRING,
              created_at STRING,
              updated_at STRING,
              ingestion_time STRING,
              batch_no STRING
              )
              USING DELTA
              CLUSTER BY(station_id,state_code,charger_type)
              LOCATION 'abfss://bronze@voltgridstorageaccount.dfs.core.windows.net/stations/'""")

In [0]:
spark.sql("""
          CREATE TABLE IF NOT EXISTS voltgrid.bronze.sessions(
              id INT,
              session_id STRING,
              vehicle_id STRING,
              station_id STRING,
              customer_id STRING,
              started_at TIMESTAMP,
              ended_at TIMESTAMP,
              duration_min INT,
              energy_kwh FLOAT,
              cost_aud FLOAT,
              peak_power_kw FLOAT,
              connector_type STRING,
              session_status STRING,
              payment_id STRING,
              created_at TIMESTAMP,
              updated_at TIMESTAMP
              )
              USING DELTA
              CLUSTER BY (session_id,vehicle_id,station_id,payment_id)
              LOCATION 'abfss://bronze@voltgridstorageaccount.dfs.core.windows.net/sessions/'""")

In [0]:
%sql
ALTER TABLE voltgrid.silver.payments
--SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name')
DROP COLUMNS(source_path)

In [0]:
%sql
ALTER TABLE voltgrid.silver.sessions
CLUSTER BY(session_id,connector_type,session_status)

In [0]:
%sql
ALTER TABLE voltgrid.silver.vehicles

In [0]:
spark.sql("""CREATE TABLE IF NOT EXISTS voltgrid.bronze.customers(
    id INT,
    customer_id STRING,
    full_name STRING,
    email STRING,
    phone STRING,
    loyalty_tier STRING,
    signup_date DATE,
    created_at DATE,
    updated_at DATE,
    ingestion_time TIMESTAMP,
    batch_no INT
    )
    USING DELTA
    CLUSTER BY (customer_id,created_at)
    LOCATION 'abfss://bronze@voltgridstorageaccount.dfs.core.windows.net/customers/'""")

In [0]:
spark.sql("""CREATE TABLE IF NOT EXISTS voltgrid.bronze.vehicles(
    id INT,
    vehicle_id STRING,
    make STRING,
    model STRING,
    year INT,
    battery_capacity_kwh STRING,
    range_km FLOAT,
    vehicle_type STRING,
    registration_state STRING,
    partner_id STRING,
    effective_from DATE,
    effective_to STRING,
    is_current STRING,
    created_at DATE,
    updated_at DATE,
    ingestion_time TIMESTAMP,
    batch_no INT
    )
    USING DELTA
    CLUSTER BY (vehicle_id, partner_id, vehicle_type, created_at)
    LOCATION 'abfss://bronze@voltgridstorageaccount.dfs.core.windows.net/vehicles/'""")

In [0]:
spark.sql("""CREATE TABLE IF NOT EXISTS voltgrid.bronze.payments(
    id STRING,
    payment_id STRING,
    session_id STRING,
    customer_id STRING,
    gateway STRING,
    amount_aud STRING,
    gst STRING,
    payment_mode STRING,
    processed_at TIMESTAMP,
    status STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    ingestion_time TIMESTAMP,
    batch_no INT
    )
    USING DELTA
    CLUSTER BY (payment_id, customer_id, status, created_at)
    LOCATION 'abfss://bronze@voltgridstorageaccount.dfs.core.windows.net/payments/'""")

In [0]:
spark.sql("SELECT COUNT(*) FROM voltgrid.bronze.payments").show()

In [0]:
spark.sql("SELECT COUNT(*) FROM voltgrid.bronze.payments").show()